# Finding the Efficiency Frontier in Bounded Weather Predictions
*Accuracy, Cost, and Diminishing Returns Across Model Complexity*

**Project Question:**

What is the most-accurate, least resource-intensive approach to predicting weather outcomes within a bounded climate region?


This project focuses on identifying a local-host friendly predictive analysis model.

To do so, a dataset with good statistical power and potentially intricate variables is an ideal intial test. Weather has evolving variables and would need a moderately intensive but flexible predictive analysis model not typically seen on local-host servers. Narrowing the variables using in a limited geographic region and isolating to a single weather phenomena makes it an ideal starting point since it reduces environmental factors and supports interpretable analysis.

Before starting exploratory analysis, this dataset was evaluated for usability, coverage, and alignment with PJ_01 goal. Sustainability was not a priority.

In [1]:
# Dataset overview: Australian Rainfall Dataset focuses on next-day rainfall prediction within Australia.

Shape: (145460, 23)

Columns:
Index(['Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation',
       'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm',
       'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
       'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am',
       'Temp3pm', 'RainToday', 'RainTomorrow'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Date           145460 non-null  object
 1   Location       145460 non-null  object
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object
 10  WindDir3pm     141232 non-null  object
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64
 18  Cloud3pm       86102 non-null   float64
 19  Temp9am        143693 non-null  float64
 20  Temp3pm        141851 non-null  float64
 21  RainToday      142199 non-null  object
 22  RainTomorrow   142193 non-null  object
dtypes: float64(16), object(7)
memory usage: 25.5+ MB

SyntaxError: invalid syntax (1648242897.py, line 5)

# Dataset selection criteria
 (please see above cell for details)

* Scale and statistical power

Dataset contains 145K obs across 23 variables wihtout exceeding 30MB of data, making it sufficient for analysis but not excessing computational overhead.  

* Clear tgt definition

Binary tgt variable (RainTomorrow) exists with minimal missingness (~2%), making it reasonably good for classification tasks.
  
* Domain-relevant feature coverage

Historically high-value indicators in rainfall prediction such as temperature, humnidity, and wind have multiple associated variables AND are largely complete with high numeric interpretability. 

* Informative* missingness patterns

Several variables (e.g. cloud cover, sunshine, evaporation) have substantial missingness, but in a pattern consistent with station-dependent measurments practices as opposed to random data loss.


 *Previous research into weather related variables, geographical data collection strategies, and variable relevancy occured prior to the above determination assessments for a usable dataset. A robust dataset is only as useful as its ability to be interpreted. 

 
# Implications for analysis strategy
 (below observations informed the structure of analysis)

* Prioritization of broad-coverage variables to minimize bias caused by uneven measurments.
* Largely missing variables are treated as secondary/comntextual indicators as opposed to primary predictors.
* Missingness* itself is considered a property of the data-collection process and may indicate geographic or instrumental effects.

*In this analysis, missing data is treated as a potential source of contextual signal.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/weather-dataset-rattle-package/weatherAUS.csv"
)

# Class balance for the target variable
counts = df['RainTomorrow'].value_counts()
proportions = df['RainTomorrow'].value_counts(normalize=True)

counts, proportions


# Baseline Performance and Class Imbalance

Class balance is now quantified (above):

Question: *Will it rain tomorrow?*


> No - 110,316 (77.58%)
>
> Yes - 31,877 (22.42%)


*This means:*
- Roughly 3 out of 4 days did not have rain tomorrow
- Rain events are meaningfully present, but clearly the minority
- The imbalance doesn't qualify as extreme, but it is enough to distort naive metrics



A model that always predicts "No Rain Tomorrow" would achieve:
> Navie accuracy = 77.6%

*Without:*
* using any weather variables
* learning any structure
* incurring any compute cost

*What this means for PJ_01:*

1. Any model that doesn't beat ~77.6% accuracy is functionally useless
2. Any model that beats it by only 1-2% may not justify additional complexity
3. Accuracy alone is insufficient because a model can score ~78% and never predict rain


**Results summarized:**

A naive classifier that always predicts the majoarity class ("No") would achieve an approximate accuracy of 77.6%, without learning any meterological relationships.

This establishes a minimum performance threshold. Any predictive model must exceed this baseline by a meaningful margin (>5%) to justify additional coputational cost or model complexity. As a result, accuracy alone isn't sufficient for evaluation. Later stages of analysis will consider additional metrics to better capture model usefulness under real-world contraints.

# Minimal-Feature Baseline

**Goal**: Beat the 77.6% naive baseline using the msallest, cheapest set of meaningful variables.

*Hypothesis being tested:* A small number of physically meaningful features can deliver most of the acheiveable predictive power in a bounded climate region.

*Desired variable characteristics:*

* widely available (low missingness)
* physically linked to rain
* cheap to measure/compute

*Proposed first set:*
> Humidity9am
>
> Humidity3pm
>
> Pressure9am
>
> Pressure3pm
>
> RainToday

*Justification:*
* Humidity -> mositure availability
* Pressure -> large-scale weather systems
* RainToday -> persistence/momentum effect
* All have relatively good coverage
* All are interpretable


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
import pandas as pd

# Select minimal feature set
features = [
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "RainToday"
]

# Subset and drop rows with missing values
df_min = df[features + ["RainTomorrow"]].dropna()

# Encode target
df_min["RainTomorrow"] = df_min["RainTomorrow"].map({"No": 0, "Yes": 1})
df_min["RainToday"] = df_min["RainToday"].map({"No": 0, "Yes": 1})

X = df_min[features]
y = df_min["RainTomorrow"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Considered "least-resource" due to no feature engineering, inputation tricks, resampling, or heavy preprocessing

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=None
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

accuracy, recall


# Minimal-Feature Baseline Results

>Accuracy = 83.65%
>
>Recall (Rain) = 44.9%

*Using:*
* 5 physical features
* Logistic regression
* No feature engineering
* No resampling
* Minimal preprocessing
* Very low computational cost

*Overall Improvements:*
* +6.0 percentage ponts accuracy
* +44.9 percentage points recall (rain)
* Close to free computational cost

*Tradeoff:*
* Model is conservative (83.6%, 44.9%)
* Avoids false positives
* Misses some rain events


**Results Summarized:**

Using a logicstic regression model with five high-coverage, physically interpretable features (humidity, pressure, and prior rainfall, the model acheieved a clear improvement over the naive baseline. 

Despite its simplicity, the model exceeds baseline accuracy (~78%) while recovering a substantial portion of rainfall events. This shows that a large share of predictive signal can be captured by a small, low-cost feature set. So far, the results support the idea that, within a bounded climate region, lightweight models paired with meaningful variables can deliver strong performance without heavy computational overhead. 

# Shallow Decision Tree Comparison

Next test will be with the *Shallow Decision Tree* since it is the smallest possible step up in complexity. If a single shallow tree doesn't meaningfully outperform logistic regression, that's a strong indicator. Since PJ_01 is testing for least-resource intensive models, a slow rise in complexity will provide the most granular assessment. 

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score

# Shallow decision tree (constrained)
tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=100,
    random_state=42
)

tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

tree_accuracy = accuracy_score(y_test, y_pred_tree)
tree_recall = recall_score(y_test, y_pred_tree)

tree_accuracy, tree_recall

# Why these constrains matter:
# max_depth=5 -> prevents memorization
# min_samples_leaf=100 -> forced generalization
# Keeps the model cheap, stable and interpretable

# Shallow Decision Tree Comparison Results

*Shallow Tree*:
> Accuracy: 83.88%
>
> Recall (Rain): 44.90%

*Logistic Regression (previous model):*
> Accuracy: 83.65%
>
> Recall (Rain): 44.90%

| Model                 | Accuracy   | Recall (Rain) | Complexity |
| --------------------- | ---------- | ------------- | ---------- |
| Naïve baseline        | 77.6%      | 0%            | trivial    |
| Logistic regression   | 83.65%     | **44.9%**     | very low   |
| Shallow decision tree | **83.88%** | 40.4%         | moderate   |

*What changed:*
* Accuracy: +0.23%
* Recall: -4.5%
* Complexity: Higher
* Interpretability: Lower

*Implications:*
* The dominant signal is largely linear
* Humidity, pressue, and persistence effects do most of the work
* Non-linear interactions exist, but they don't significantly improve next-day rain detection at this scale
* The decision boundary appears smooth as opposed to jagged
* Overall, the system rewards good variables more than complex structures at this stage


**Results summarized:**

A shallow decision tree was evaluated to assess whether modest non-linearity improves predictive performance beyond the linear baseline. While the tree acheived a slight increase in overall accuracy (~0.2%), it underperformed logistic regression in recall for rainfall events. This indicates that additional model expressiveness doesn't meaningfully imnprove detection for rain outcomes under the tested constraints. 

These results suggest that the efficiency frontier for this tast likes near the linear baseline, with diminishing returns from increased model complexity.

# Part 2
# Increased Experiment: Efficiency Frontier Sweep

*Goal*: Determine whether moderate model complexity yields meaningful gains over the lightweight logistic baseline. 

*Hypothesis being tested*: A moderate model complexity will provide greater accuracy and recall over a lightweight model.

*What constitutes a "moderate model" within the contraints of "most-accurate, least resource-intensive"*:
* More expressive than linear
* Fast to train and infer
* Works well with tabular data
* Runs well on local machine
* Doesn't require heavy featre engineering or GPUs

*Types of "moderate models"*:
* Logisitic Regression (previously used)
* Shallow Decision Tree
* Random Forest (constrained)
* Gradient Boosting
* XGBoost/LightGBM/CatBoost
* Support Vector Machine (RBF kernel)

*Experiment*:

Fairly test four models with increasing complexity to evaluate the accuracy-to-cost ratio.

Run a small benchmark grid across four model families from simplest to complex while keeping the below:

* same train/test split
* same feature set initally (to isolate model effect)
* same evaluation metrics
* training time (and optional model size)

*Chosen Models*:
1. Logistic Regression (already done)
   * Everything is judged relative to this
   * Low capacity | Tests whether linear structure + rich features is enough
3. Shallow Decision Tree (depths = 3, 5, 8)
   * Smallest increase in complexity
   * Adds limited non-linearlity | Tests whether simple interactions matter
4. Random Forest
   * Tests if ensemble averging is cost effective
   * Adds ensemble averaging | Tests variance reduction benefits
6. Gradient Boosting
   * Often gives best accuracy among models
   * Adds sequential error correction | Is fine-grained structure worth the cost

*Recorded Metrics*:
* Accuracy
* Recall (rain=1)
* Precision (rain=1)
* F1
* Training time (seconds)

***Requirment: All models are given access to the same meaningful information.***

With these in mind, ***PJ_01 will re-run the Logistical Regression and Shallow Decision Tree models*** including the additional variables "location" and "wind directions".

This will be done to prevent biases towards models that prefer numeric-only inputs and accurately test all four models as fairly as possible within the confines of the dataset.


Summary:

To avoid numerical-feature bias, models of increasing complexity will be evaluated using a shared preprocessing pipeline and expanded feature set that include geographic and directional context, allowing performance gains to be attriubuted to model capacity rather than information asymmetry. 

In [ ]:
# Run this once for experiment set up
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier


# Logistic Regression with Expanded Variables

*Goal*: Establish new baseline when all models are given the same information (numerical + categorical), to fairly measure how much accuracy complexity gains.

Side Question: Is the signal itself mostly linear even with richer context provided?

Differences from first model run:
* Including "Location" and "Wind Directions (categorical)"
* Using imputation instead of dropping rows
* Using one-hot encoding
* Keeping everything inside a single pipeline

Why:
*  No data leakage
*  Fair comparison with trees/forest
*  Reproducibility
*  Better efficiency measurement



In [ ]:
# what does every model get to see

target = "RainTomorrow"

numeric_features = [
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "MinTemp", "MaxTemp",
    "WindSpeed9am", "WindSpeed3pm",
    "Rainfall"
]

categorical_features = [
    "RainToday",
    "Location",
    "WindGustDir", "WindDir9am", "WindDir3pm"
]

# how to avoid leakage and missing-target issues

df_model = df[numeric_features + categorical_features + [target]].copy()

# Drop rows where the target is missing
df_model = df_model.dropna(subset=[target])

# Encode target
df_model[target] = df_model[target].map({"No": 0, "Yes": 1})

X = df_model[numeric_features + categorical_features]
y = df_model[target]

# ensures generalization

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# treat numeric and categorical data fairly

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# what if we add info but not complexity

logreg = LogisticRegression(
    max_iter=3000,
    solver="lbfgs"
)

# what was the performance and what did it cost?

pipe_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", logreg)
])

start = time.time()
pipe_logreg.fit(X_train, y_train)
train_time = time.time() - start

y_pred = pipe_logreg.predict(X_test)

results_logreg = {
    "model": "LogReg_expanded",
    "train_s": train_time,
    "accuracy": accuracy_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred)
}

results_logreg


# Logistic Regression with Expanded Variables Results

*Results:*
> Accuracy   ≈ 83.7%
>
> Recall     ≈ 44.8%
>
> Precision  ≈ 71.6%
>
> F1         ≈ 0.551
>
> Train time ≈ 50.3s

*What changed:*
| Model                 | Accuracy   | Recall    | Precision | Notes         |
| --------------------- | ---------- | --------- | --------- | ------------- |
| Minimal-feature LR    | ~83.65%    | **44.9%** | -        | numeric-only  |
| Shallow tree (old)    | ~83.88%    | 40.4%     | -         | numeric-only  |
| **Expanded LR (now)** | **83.7%** | **44.8%** | **71.8%** | +categoricals |

This seems to indicate that adding categoricals didn't materially change the performance for logistic regression.

**Results Summarized:**

Logistic regression was re-evaluated using an expanded feature set that included geographical location and wind-direction variables, with unified preprocessing applied. Despite the additional information, the performance remained largedly unchanged suggesting the dominate predictive signal is already captured by core physical variables, and additional contextual information doesn't substantially improve a linear decision boundary.

# Shallow Decision Tree with Expanded Variables

*Goal*: Test if limited non-linearity is a better cost when categorical context is added.

Max_depth = 3, 5, 8

In [ ]:
tree_results = []

# where depth is added and constrained
for depth in [3, 5, 8]:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=100,
        random_state=42
    )
    
    pipe_tree = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", tree)
    ])
    
    start = time.time()
    pipe_tree.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_pred = pipe_tree.predict(X_test)
    
    tree_results.append({
        "model": f"Tree_depth{depth}_expanded",
        "train_s": train_time,
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred)
    })

pd.DataFrame(tree_results)


# Shallow Decision Tree with Expanded Variables Results

*Results*:

> | Model               | Train (s) |   Accuracy |     Recall |  Precision |         F1 |
| ------------------- | --------: | ---------: | ---------: | ---------: | ---------: |
| **LogReg_expanded** |  **50.7** | **0.8369** | **0.4475** | **0.7188** | **0.5516** |
| Tree_depth3         |      0.70 |     0.8283 |     0.3490 |     0.7519 |     0.4768 |
| Tree_depth5         |      0.96 |     0.8329 |     0.3900 |     0.7425 |     0.5114 |
| Tree_depth8         |      2.02 |     0.8354 |     0.4290 |     0.7241 |     0.5388 |

*What this means*:

Tree_depth8 results were the closest to Logistic Regression, but still behind implying there are non-linear effects but that they're weak in relation to the dominant linear signal. This idicates the system is mostly influenced by smooth, monotonic relationships, something that's a property of the data-generating process, not the models.

**Results Summarized:**

Trees improve with depth but asympotically. Even at depth 8, the Shallow Decision Tree model doesn't beat the Logistic Regression model on recall, accuracy or F1. While this model took significantly less training time comparatively, that's only one cateory and not a heavily valued one where the key comparision is performance per unit complexity as opposed to raw seconds. Based on these results, it can be surmized that adding non-linearity and categorical context doesn't surpass a linear model with the same information.


# Random Forest (Constrained) with Expanded Variables

*Goal*: Test whether ensemble averaging unlocks additional signal and if stronger models plateau or signficantly improve.

*Why test this*:

* A Random Forest model is the strongest remaining classical model

* If it doesn't beat the Logisitc Tegression model meaningfully, then the overall conclusion is solidified

* At least one Random Forest model test provides context without expanding the testing into hyperparameter fishing

With the goal of using "moderate complexity", the below will be used:


> 200 trees -> enough to stabilize
>
> max_depth= 10 -> allows interactions, prevents memorization
>
> min_samples_leaf= 50 -> controls varience with one-hot features
>
> n_jobs= -1 -> fair use of CPU while still local-host friendly


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

pipe_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

start = time.time()
pipe_rf.fit(X_train, y_train)
train_time = time.time() - start

y_pred = pipe_rf.predict(X_test)

rf_results = {
    "model": "RandomForest_expanded",
    "train_s": train_time,
    "accuracy": accuracy_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred)
}

rf_results


# Random Forest (Constrained) with Expanded Variables Results

Results:

>Accuracy = 83.43%
>
>Recall = 35.71%
>
>Precision = 78.81%
>
>F1 = .492

*What this means*:
The Random Forest model lost on recall, accuracy, F1 and added complexity without a meaningful increased benefit.

Results summarized:

The efficiency peak remains at/near the regularized Logistic Regression model. 

# Key Findings

*Project Question:* 

What is the most-accurate, least resource-intensive approach to predicting weather outcomes within a bounded climate region?

Overall results:
| Model                        |   Accuracy |    Recall | Precision |        F1 | Train Time |
| ---------------------------- | ---------: | --------: | --------: | --------: | ---------: |
| Naïve baseline               |      77.6% |        0% |         — |         — |         ~0 |
| LogReg (minimal)             |     ~83.6% | **44.9%** |         — |         — |   very low |
| LogReg (expanded)            | **83.69%** | **44.8%** |     71.8% | **0.551** |      50.2s |
| Tree depth 3 (expanded)      |     82.83% |     34.9% |     75.2% |     0.477 |       0.7s |
| Tree depth 5 (expanded)      |     83.29% |     39.0% |     74.3% |     0.511 |       1.0s |
| Tree depth 8 (expanded)      |     83.54% |     42.9% |     72.4% |     0.539 |       2.0s |
| **Random Forest (expanded)** | **83.44%** | **35.7%** | **78.8%** | **0.492** |      13.7s |

*Results Summarized*:

Using multiple model families and feature sets, results indicate a regularized linear model using a small number of physically meaningful, high-coverage variables achieves near-optimal predictive performance with the least resources needed. Increasing model complexity yields diminishing or negative returns relative to added computational cost. In regards to bounded climate regions, predictive efficiency is maximized by feature selection on interpretability rather than model expressivness. 


*Why does this matter*: 

From an applied perspective, these results suggest that for short-horizon, weather-dependent decisions, commercial value is maximized by feature relevance, interpretability, and computational efficiency. This gives PJ_01 direct implications for sectors where forecasts must be fast, local, and explainable.

In commercial settings, weather preditions need to provide data for timely decisions, operational thresholds, cost asymmetry, and reliability under constraints. This test establishes that for next-day weather outcomes in a bounded climate region, most usable predictive value comes from a small set of physically meaningful variables, and not necessarily increasing model complexity. 

**These results indicate possible cheaper alternatives for weather-oriented decision horizons and deliverables.**

Essentially, this test indicates that the correct variables can be used with cheap, simple logistic regression models and still provide meaningful data over expensive and complex models. 

***Sectors Impacted:*** 

* Agriculture & Agric-Logistics
* Energy (Solar/Wind/Grid Ops)
* Insurance & Risk Assessments
* Transportation & Logistics
* Construction & Field Operations